In [7]:
import json
import pandas as pd
import re
from pathlib import Path
from typing import Dict, Any, Optional, Sequence, Union
from collections import defaultdict
from datetime import datetime


class DiabetesDatasetTransformer:
    """
    Transformer for Manchester Diabetes Study dataset.
    
    Maps CSV files to Sensor-Augmented OCEL format:
    - Sensor Events: BloodGlucose (BG), Sleep (S)
    - Behavior Events: BasalInsulin (BaI), BolusInsulin (BoI), Meal (M), Activity (A), SleepTime (ST)
    - Objects: Participant (P)
    """
    
    ATTRIBUTE_TIME_PLACEHOLDER = "__PENDING_TIME__"
    
    def __init__(self, data_dir: Path):
        """
        Initialize transformer with data directory.
        
        Args:
            data_dir: Path to the dataset directory
        """
        self.data_dir = Path(data_dir)
        self.participants = {}  # participant_id -> object_id
        self.event_counter = defaultdict(int)  # event_type -> counter for unique IDs
        self.object_index: Dict[str, Dict[str, Any]] = {}
        self.object_relationship_cache: Dict[str, set] = defaultdict(set)
        
        # OCEL output structures
        self.sensor_event_types = [
            {
                'name': 'BloodGlucose',
                'attributes': [
                    {'name': 'value', 'type': 'number'},
                    {'name': 'unit', 'type': 'string'}
                ]
            },
            {
                'name': 'Sleep',
                'attributes': [
                    {'name': 'heart_rate', 'type': 'number'},
                    {'name': 'current_activity_type_intensity', 'type': 'number'},
                    {'name': 'stress_level_value', 'type': 'number'},
                    {'name': 'steps', 'type': 'number'},
                    {'name': 'sleep_level', 'type': 'number'},
                    {'name': 'resting_heart_rate', 'type': 'number'}
                ]
            }
        ]
        
        self.behavior_event_types = [
            {
                'name': 'BasalInsulin',
                'attributes': [
                    {'name': 'basal_dose', 'type': 'number'},
                    {'name': 'basal_unit', 'type': 'string'},
                    {'name': 'insulin_kind', 'type': 'string'}
                ]
            },
            {
                'name': 'BolusInsulin',
                'attributes': [
                    {'name': 'bolus_dose', 'type': 'number'},
                    {'name': 'bolus_unit', 'type': 'string'}
                ]
            },
            {
                'name': 'Meal',
                'attributes': [
                    {'name': 'meal_type', 'type': 'string'},
                    {'name': 'meal_tag', 'type': 'string'},
                    {'name': 'carbs_g', 'type': 'number'},
                    {'name': 'prot_g', 'type': 'number'},
                    {'name': 'fat_g', 'type': 'number'},
                    {'name': 'fiber_g', 'type': 'number'}
                ]
            },
            {
                'name': 'Activity',
                'attributes': [
                    {'name': 'activity_type', 'type': 'string'},
                    {'name': 'activity_Kcal', 'type': 'number'},
                    {'name': 'step_count', 'type': 'number'},
                    {'name': 'distance_m', 'type': 'number'},
                    {'name': 'duration_s', 'type': 'number'},
                    {'name': 'active_time_s', 'type': 'number'},
                    {'name': 'start_time_s', 'type': 'number'},
                    {'name': 'start_time_offset_s', 'type': 'number'},
                    {'name': 'met', 'type': 'number'},
                    {'name': 'intensity', 'type': 'string'},
                    {'name': 'motion_intensity_mean', 'type': 'number'},
                    {'name': 'motion_intensity_max', 'type': 'number'}
                ]
            },
            {
                'name': 'SleepTime',
                'attributes': [
                    {'name': 'calendar_date', 'type': 'string'},
                    {'name': 'duration_in_sec', 'type': 'number'},
                    {'name': 'start_time_offset_s', 'type': 'number'},
                    {'name': 'unmeasurable_sleep_s', 'type': 'number'},
                    {'name': 'deep_sleep_s', 'type': 'number'},
                    {'name': 'light_sleep_s', 'type': 'number'},
                    {'name': 'rem_sleep_s', 'type': 'number'},
                    {'name': 'awake_s', 'type': 'number'},
                    {'name': 'sleep_levels_map_deep', 'type': 'string'},
                    {'name': 'sleep_levels_map_light', 'type': 'string'},
                    {'name': 'sleep_levels_map_awake', 'type': 'string'},
                    {'name': 'sleep_levels_map_rem', 'type': 'string'},
                    {'name': 'sleep_levels_map_unmeasurable', 'type': 'string'},
                    {'name': 'validation', 'type': 'string'}
                ]
            }
        ]
        
        self.object_types = [
            {
                'name': 'Participant',
                'attributes': [
                    {'name': 'ID', 'type': 'string'},
                    {'name': 'weight_kg', 'type': 'number'},
                    {'name': 'height_m', 'type': 'number'},
                    {'name': 'bmi', 'type': 'number'}
                ]
            }
        ]
        
        self.sensor_events = []
        self.behavior_events = []
        self.objects = []
    
    def extract_participant_id(self, filename: str) -> Optional[str]:
        """
        Extract participant ID from filename.
        
        Args:
            filename: CSV filename (e.g., 'UoMActivity2301.csv', 'UoMGlucose2301.csv')
            
        Returns:
            Participant ID (e.g., '2301') or None
        """
        # Match UoM followed by any characters, then 4 digits
        match = re.search(r'UoM.*?(\d{4})', filename)
        return match.group(1) if match else None
    
    def get_or_create_participant(self, participant_id: str) -> str:
        """
        Get or create participant object.
        
        Args:
            participant_id: Participant ID
            
        Returns:
            Participant object ID (uses participant_id as ID)
        """
        if participant_id not in self.participants:
            # Use prefixed participant ID as object ID
            object_id = f"P_{participant_id}"
            participant_obj = {
                'id': object_id,
                'type': 'Participant',
                'attributes': [
                    {
                        'name': 'ID',
                        'value': participant_id,
                        'time': self.ATTRIBUTE_TIME_PLACEHOLDER
                    }
                ],
                'relationships': []
            }
            self.objects.append(participant_obj)
            self.object_index[object_id] = participant_obj
            self.participants[participant_id] = object_id
        return self.participants[participant_id]
    
    def _get_event_id(self, event_type: str) -> str:
        """
        Generate a readable event ID with abbreviation prefix.
        
        Args:
            event_type: Event type name
            
        Returns:
            Event ID (e.g., 'BG_1', 'ST_2', 'BaI_1', 'BoI_1')
        """
        # Map event types to abbreviations
        abbreviations = {
            'BloodGlucose': 'BG',
            'Sleep': 'S',
            'BasalInsulin': 'BaI',
            'BolusInsulin': 'BoI',
            'Meal': 'M',
            'Activity': 'A',
            'SleepTime': 'ST'
        }
        
        abbrev = abbreviations.get(event_type, 'EV')
        
        # Initialize counter if needed
        self.event_counter[event_type] += 1
        return f"{abbrev}_{self.event_counter[event_type]}"
    
    def _coerce_value(self, value: Any, cast) -> Optional[Any]:
        if value is None:
            return None
        try:
            if pd.isna(value):
                return None
        except TypeError:
            pass
        try:
            return cast(value)
        except (TypeError, ValueError):
            return None

    def _as_str(self, value: Any) -> Optional[str]:
        if value is None:
            return None
        try:
            if pd.isna(value):
                return None
        except TypeError:
            pass
        return str(value)

    def _build_attributes(self, attributes: Dict[str, Any]) -> Any:
        cleaned = []
        for name, value in attributes.items():
            if value is None:
                continue
            try:
                if pd.isna(value):
                    continue
            except TypeError:
                pass
            cleaned.append({'name': name, 'value': value})
        return cleaned

    def _prepare_dataframe(
        self,
        df: pd.DataFrame,
        timestamp_columns: Union[str, Sequence[str]],
        *,
        dayfirst: bool = True
    ) -> pd.DataFrame:
        if isinstance(timestamp_columns, (str, bytes)):
            timestamp_columns = (timestamp_columns,)
        timestamps = None
        for column in timestamp_columns:
            if column not in df:
                continue
            ts = pd.to_datetime(df[column], errors='coerce', dayfirst=dayfirst)
            timestamps = ts if timestamps is None else timestamps.fillna(ts)
        if timestamps is None:
            return df.iloc[0:0]
        df = df.assign(_timestamp=timestamps)
        df = df[~df['_timestamp'].isna()].copy()
        df['_timestamp'] = df['_timestamp'].apply(
            lambda ts: ts.to_pydatetime() if isinstance(ts, pd.Timestamp) else ts
        )
        return df

    def _get_raw_timestamp(
        self,
        record: Dict[str, Any],
        timestamp_columns: Union[str, Sequence[str]]
    ) -> Optional[str]:
        if isinstance(timestamp_columns, (str, bytes)):
            timestamp_columns = (timestamp_columns,)
        for column in timestamp_columns:
            raw_value = self._as_str(record.get(column))
            if raw_value is not None:
                return raw_value
        return None

    def _normalize_timestamp(self, raw_value: Optional[str]) -> Optional[str]:
        if raw_value is None:
            return None
        parsed = self.parse_timestamp(raw_value)
        if parsed is None:
            return None
        if parsed.tzinfo is not None:
            parsed = parsed.replace(tzinfo=None)
        return parsed.strftime('%Y-%m-%dT%H:%M:%S')

    def _add_object_relationship(self, object_id: str, target_id: str, qualifier: str) -> None:
        """Ensure relationships are stored on both sides."""
        participant_obj = self.object_index.get(object_id)
        if not participant_obj:
            return
        relationships = participant_obj.setdefault('relationships', [])
        cache = self.object_relationship_cache[object_id]
        cache_key = (target_id, qualifier)
        if cache_key in cache:
            return
        relationships.append({
            'objectId': target_id,
            'qualifier': qualifier
        })
        cache.add(cache_key)
    
    def _finalize_participant_attribute_times(self) -> None:
        """Replace placeholder attribute times with the earliest related event timestamp."""
        placeholder = self.ATTRIBUTE_TIME_PLACEHOLDER
        participant_earliest: Dict[str, datetime] = {}
        
        for event in self.sensor_events + self.behavior_events:
            time_value = event.get('time')
            if not time_value:
                continue
            if isinstance(time_value, datetime):
                timestamp = time_value
            else:
                timestamp = self.parse_timestamp(time_value)
            if timestamp is None:
                continue
            
            for rel in event.get('relationships', []):
                participant_id = rel.get('objectId')
                if not participant_id:
                    continue
                current = participant_earliest.get(participant_id)
                if current is None or timestamp < current:
                    participant_earliest[participant_id] = timestamp
        
        fallback_time = datetime.now()
        
        for obj in self.objects:
            obj_time = participant_earliest.get(obj['id'], fallback_time)
            for attr in obj.get('attributes', []):
                if attr.get('time') == placeholder:
                    attr['time'] = obj_time.isoformat()
    
    def parse_timestamp(self, value: Any) -> Optional[datetime]:
        """Parse timestamp from various formats."""
        if pd.isna(value):
            return None
        
        if isinstance(value, datetime):
            return value
        
        if isinstance(value, pd.Timestamp):
            return value.to_pydatetime()
        
        # Try common formats
        formats = [
            '%d/%m/%Y %H:%M',
            '%d/%m/%Y %H:%M:%S',
            '%Y-%m-%d %H:%M:%S',
            '%Y-%m-%dT%H:%M:%S',
            '%Y-%m-%d %H:%M',
            '%Y-%m-%d',
        ]
        
        for fmt in formats:
            try:
                return datetime.strptime(str(value), fmt)
            except ValueError:
                continue
        
        # Try pandas parsing with dayfirst=True for %d/%m/%Y format
        try:
            return pd.to_datetime(value, dayfirst=True).to_pydatetime()
        except:
            return None
    
    def create_sensor_event(
        self,
        event_type: str,
        timestamp: datetime,
        attributes: Dict[str, Any],
        participant_id: str
    ) -> None:
        """Create a sensor event."""
        event_id = self._get_event_id(event_type)
        participant_obj_id = self.get_or_create_participant(participant_id)
        
        event_attributes = self._build_attributes(attributes)
        if timestamp is None:
            return
        if isinstance(timestamp, pd.Timestamp):
            time_value = timestamp.isoformat()
        elif isinstance(timestamp, datetime):
            time_value = timestamp.isoformat()
        else:
            time_value = str(timestamp)
        if not time_value:
            return
        
        event = {
            'id': event_id,
            'sensorEventType': event_type,
            'time': time_value,
            'sensorEventTypeAttributes': event_attributes,
        }

        relationships = []
        if participant_obj_id:
            relationships.append({
                'objectId': participant_obj_id,
                'qualifier': 'corr'
            })

        if relationships:
            event['relationships'] = relationships
        
        self.sensor_events.append(event)
        if participant_obj_id:
            self._add_object_relationship(participant_obj_id, event_id, 'corr')
    
    def create_behavior_event(
        self,
        event_type: str,
        timestamp: datetime,
        attributes: Dict[str, Any],
        participant_id: str
    ) -> None:
        """Create a behavior event."""
        event_id = self._get_event_id(event_type)
        participant_obj_id = self.get_or_create_participant(participant_id)
        
        event_attributes = self._build_attributes(attributes)
        if timestamp is None:
            return
        if isinstance(timestamp, pd.Timestamp):
            time_value = timestamp.isoformat()
        elif isinstance(timestamp, datetime):
            time_value = timestamp.isoformat()
        else:
            time_value = str(timestamp)
        if not time_value:
            return
        
        event = {
            'id': event_id,
            'behaviorEventType': event_type,
            'time': time_value,
            'behaviorEventTypeAttributes': event_attributes,
        }

        relationships = []
        if participant_obj_id:
            relationships.append({
                'objectId': participant_obj_id,
                'qualifier': 'corr'
            })

        if relationships:
            event['relationships'] = relationships
        
        self.behavior_events.append(event)
        if participant_obj_id:
            self._add_object_relationship(participant_obj_id, event_id, 'corr')
    
    def process_glucose_data(self) -> None:
        """Process glucose CSV files."""
        for filepath in self.data_dir.glob('**/Glucose Data/*.csv'):
            print(f"  -> Loading glucose file: {filepath}")
            participant_id = self.extract_participant_id(filepath.name)
            if not participant_id:
                print(f"Warning: Could not extract participant ID from {filepath}")
                continue
            
            df = pd.read_csv(filepath)
            df = self._prepare_dataframe(df, 'bg_ts')
            if df.empty:
                continue

            for record in df.to_dict('records'):
                raw_timestamp = self._get_raw_timestamp(record, 'bg_ts')
                normalized_timestamp = self._normalize_timestamp(raw_timestamp)
                if normalized_timestamp is None:
                    continue
                record.pop('_timestamp', None)
                attributes = {
                    'value': self._coerce_value(record.get('value'), float),
                    'unit': 'mmol/L'
                }

                self.create_sensor_event('BloodGlucose', normalized_timestamp, attributes, participant_id)
    
    def process_sleep_sensor_data(self) -> None:
        """Process sleep sensor CSV files (UoMsleep*.csv)."""
        for filepath in self.data_dir.glob('**/Sleep Data/UoMsleep*.csv'):
            participant_id = self.extract_participant_id(filepath.name)
            if not participant_id:
                continue
            
            df = pd.read_csv(filepath)
            df = self._prepare_dataframe(df, 'sleep_ts')
            if df.empty:
                continue

            for record in df.to_dict('records'):
                raw_timestamp = self._get_raw_timestamp(record, 'sleep_ts')
                normalized_timestamp = self._normalize_timestamp(raw_timestamp)
                if normalized_timestamp is None:
                    continue
                record.pop('_timestamp', None)
                attributes = {
                    'heart_rate': self._coerce_value(record.get('heart_rate'), float),
                    'current_activity_type_intensity': self._coerce_value(record.get('current_activity_type_intensity'), float),
                    'stress_level_value': self._coerce_value(record.get('stress_level_value'), float),
                    'steps': self._coerce_value(record.get('step_count'), int),
                    'sleep_level': self._coerce_value(record.get('sleep_level'), int),
                    'resting_heart_rate': self._coerce_value(record.get('resting_heart_rate'), float)
                }

                self.create_sensor_event('Sleep', normalized_timestamp, attributes, participant_id)
    
    def process_basal_insulin_data(self) -> None:
        """Process basal insulin CSV files."""
        for filepath in self.data_dir.glob('**/Insulin Data/Basal Data/*.csv'):
            participant_id = self.extract_participant_id(filepath.name)
            if not participant_id:
                continue
            
            df = pd.read_csv(filepath)
            df = self._prepare_dataframe(df, 'basal_ts')
            if df.empty:
                continue

            for record in df.to_dict('records'):
                raw_timestamp = self._get_raw_timestamp(record, 'basal_ts')
                normalized_timestamp = self._normalize_timestamp(raw_timestamp)
                if normalized_timestamp is None:
                    continue
                record.pop('_timestamp', None)
                attributes = {
                    'basal_dose': self._coerce_value(record.get('basal_dose'), float),
                    'basal_unit': 'U',
                    'insulin_kind': self._as_str(record.get('insulin_kind'))
                }

                self.create_behavior_event('BasalInsulin', normalized_timestamp, attributes, participant_id)
    
    def process_bolus_insulin_data(self) -> None:
        """Process bolus insulin CSV files."""
        for filepath in self.data_dir.glob('**/Insulin Data/Bolus Data/*.csv'):
            participant_id = self.extract_participant_id(filepath.name)
            if not participant_id:
                continue
            
            df = pd.read_csv(filepath)
            df = self._prepare_dataframe(df, 'bolus_ts')
            if df.empty:
                continue

            for record in df.to_dict('records'):
                raw_timestamp = self._get_raw_timestamp(record, 'bolus_ts')
                normalized_timestamp = self._normalize_timestamp(raw_timestamp)
                if normalized_timestamp is None:
                    continue
                record.pop('_timestamp', None)
                attributes = {
                    'bolus_dose': self._coerce_value(record.get('bolus_dose'), float),
                    'bolus_unit': 'U'
                }

                self.create_behavior_event('BolusInsulin', normalized_timestamp, attributes, participant_id)
    
    def process_meal_data(self) -> None:
        """Process nutrition/meal CSV files."""
        for filepath in self.data_dir.glob('**/Nutrition Data/*.csv'):
            participant_id = self.extract_participant_id(filepath.name)
            if not participant_id:
                continue
            
            df = pd.read_csv(filepath)
            df = self._prepare_dataframe(df, 'meal_ts')
            if df.empty:
                continue

            for record in df.to_dict('records'):
                raw_timestamp = self._get_raw_timestamp(record, 'meal_ts')
                normalized_timestamp = self._normalize_timestamp(raw_timestamp)
                if normalized_timestamp is None:
                    continue
                record.pop('_timestamp', None)
                attributes = {
                    'meal_type': self._as_str(record.get('meal_type')),
                    'meal_tag': self._as_str(record.get('meal_tag')),
                    'carbs_g': self._coerce_value(record.get('carbs_g'), float),
                    'prot_g': self._coerce_value(record.get('prot_g'), float),
                    'fat_g': self._coerce_value(record.get('fat_g'), float),
                    'fiber_g': self._coerce_value(record.get('fibre_g'), float)
                }

                self.create_behavior_event('Meal', normalized_timestamp, attributes, participant_id)
    
    def process_activity_data(self) -> None:
        """Process activity CSV files."""
        for filepath in self.data_dir.glob('**/Activity Data/*.csv'):
            participant_id = self.extract_participant_id(filepath.name)
            if not participant_id:
                continue
            
            df = pd.read_csv(filepath)
            df = self._prepare_dataframe(df, 'activity_ts')
            if df.empty:
                continue

            for record in df.to_dict('records'):
                raw_timestamp = self._get_raw_timestamp(record, 'activity_ts')
                normalized_timestamp = self._normalize_timestamp(raw_timestamp)
                if normalized_timestamp is None:
                    continue
                record.pop('_timestamp', None)
                attributes = {
                    'activity_type': self._as_str(record.get('activity_type')),
                    'activity_Kcal': self._coerce_value(record.get('active_Kcal'), float),
                    'step_count': self._coerce_value(record.get('step_count'), int),
                    'distance_m': self._coerce_value(record.get('distance_m'), float),
                    'duration_s': self._coerce_value(record.get('duration_s'), int),
                    'active_time_s': self._coerce_value(record.get('active_time_s'), int),
                    'start_time_s': self._coerce_value(record.get('start_time_s'), int),
                    'start_time_offset_s': self._coerce_value(record.get('start_time_offset_s'), int),
                    'met': self._coerce_value(record.get('met'), float),
                    'intensity': self._as_str(record.get('intensity')),
                    'motion_intensity_mean': self._coerce_value(record.get('motion_intensity_mean'), float),
                    'motion_intensity_max': self._coerce_value(record.get('motion_intensity_max'), int)
                }

                self.create_behavior_event('Activity', normalized_timestamp, attributes, participant_id)
    
    def process_sleep_behavior_data(self) -> None:
        """Process sleep behavior CSV files (UoM*sleeptime.csv)."""
        for filepath in self.data_dir.glob('**/Sleep Data/UoM*sleeptime.csv'):
            participant_id = self.extract_participant_id(filepath.name)
            if not participant_id:
                continue
            
            df = pd.read_csv(filepath)
            df = self._prepare_dataframe(df, ('start_date_ts', 'calendar_date'))
            if df.empty:
                continue

            for record in df.to_dict('records'):
                raw_timestamp = self._get_raw_timestamp(record, ('start_date_ts', 'calendar_date'))
                normalized_timestamp = self._normalize_timestamp(raw_timestamp)
                if normalized_timestamp is None:
                    continue
                record.pop('_timestamp', None)

                attributes = {
                    'calendar_date': self._as_str(record.get('calendar_date')),
                    'duration_in_sec': self._coerce_value(record.get('duration_in_sec'), int),
                    'start_time_offset_s': self._coerce_value(record.get('start_time_offset_s'), int),
                    'unmeasurable_sleep_s': self._coerce_value(record.get('unmeasurable_sleep_s'), int),
                    'deep_sleep_s': self._coerce_value(record.get('deep_sleep_s'), int),
                    'light_sleep_s': self._coerce_value(record.get('light_sleep_s'), int),
                    'rem_sleep_s': self._coerce_value(record.get('rem_sleep_s'), int),
                    'awake_s': self._coerce_value(record.get('awake_s'), int),
                    'sleep_levels_map_deep': self._as_str(record.get('sleep_levels_map.deep')),
                    'sleep_levels_map_light': self._as_str(record.get('sleep_levels_map_light')),
                    'sleep_levels_map_awake': self._as_str(record.get('sleep_levels_map_awake')),
                    'sleep_levels_map_rem': self._as_str(record.get('sleep_levels_map_rem')),
                    'sleep_levels_map_unmeasurable': self._as_str(record.get('sleep_levels_map_unmeasurable')),
                    'validation': self._as_str(record.get('validation'))
                }
                
                self.create_behavior_event('SleepTime', normalized_timestamp, attributes, participant_id)
    
    def process_demographics_data(self) -> None:
        """Process demographics CSV files."""
        for filepath in self.data_dir.glob('**/Demographics/*.csv'):
            df = pd.read_csv(filepath)
            
            for _, row in df.iterrows():
                # Extract participant ID from "UoM 2301" format
                participant_id_str = str(row['participant_id'])
                match = re.search(r'UoM\s*(\d{4})', participant_id_str)
                participant_id = match.group(1) if match else None
                
                if not participant_id:
                    continue
                
                # Get or create participant
                object_id = self.get_or_create_participant(participant_id)
                
                # Update participant attributes
                participant_obj = self.object_index[object_id]
                # Add/update attributes
                existing_attrs = {attr['name']: attr for attr in participant_obj['attributes']}
                
                weight_val = float(row['weight_kg']) if not pd.isna(row['weight_kg']) else None
                height_val = float(row['height_m']) if not pd.isna(row['height_m']) else None
                bmi_val = float(row['bmi']) if not pd.isna(row['bmi']) else None

                if 'weight_kg' not in existing_attrs and weight_val is not None:
                    participant_obj['attributes'].append({
                        'name': 'weight_kg',
                        'value': weight_val,
                        'time': self.ATTRIBUTE_TIME_PLACEHOLDER
                    })
                if 'height_m' not in existing_attrs and height_val is not None:
                    participant_obj['attributes'].append({
                        'name': 'height_m',
                        'value': height_val,
                        'time': self.ATTRIBUTE_TIME_PLACEHOLDER
                    })
                if 'bmi' not in existing_attrs and bmi_val is not None:
                    participant_obj['attributes'].append({
                        'name': 'bmi',
                        'value': bmi_val,
                        'time': self.ATTRIBUTE_TIME_PLACEHOLDER
                    })
    
    def transform(self) -> Dict[str, Any]:
        """
        Transform all data files to OCEL format.
        
        Returns:
            Dictionary in Sensor-Augmented OCEL format
        """
        print("Processing demographics...")
        self.process_demographics_data()
        
        print("Processing glucose data...")
        self.process_glucose_data()
        
        print("Processing sleep sensor data...")
        self.process_sleep_sensor_data()
        
        print("Processing basal insulin data...")
        self.process_basal_insulin_data()
        
        print("Processing bolus insulin data...")
        self.process_bolus_insulin_data()
        
        print("Processing meal data...")
        self.process_meal_data()
        
        print("Processing activity data...")
        self.process_activity_data()
        
        print("Processing sleep behavior data...")
        self.process_sleep_behavior_data()
        
        print("Finalizing participants' attributes timing...")
        self._finalize_participant_attribute_times()
        
        return {
            'sensorEventTypes': self.sensor_event_types,
            'behaviorEventTypes': self.behavior_event_types,
            'objectTypes': self.object_types,
            'sensorEvents': self.sensor_events,
            'behaviorEvents': self.behavior_events,
            'objects': self.objects
        }
    
    def describe_output(self) -> Dict[str, Any]:
        """
        Generate a comprehensive description of the transformed output data.
        
        Returns:
            Dictionary containing statistics and descriptions of the output data
        """
        description = {
            'summary': {
                'total_sensor_events': len(self.sensor_events),
                'total_behavior_events': len(self.behavior_events),
                'total_events': len(self.sensor_events) + len(self.behavior_events),
                'total_objects': len(self.objects),
                'total_participants': len(self.participants)
            },
            'sensor_event_types': {},
            'behavior_event_types': {},
            'participants': {},
            'time_range': {
                'earliest_event': None,
                'latest_event': None,
                'time_span_days': None
            }
        }
        
        # Count sensor events by type
        for event in self.sensor_events:
            event_type = event.get('sensorEventType', 'Unknown')
            if event_type not in description['sensor_event_types']:
                description['sensor_event_types'][event_type] = 0
            description['sensor_event_types'][event_type] += 1
        
        # Count behavior events by type
        for event in self.behavior_events:
            event_type = event.get('behaviorEventType', 'Unknown')
            if event_type not in description['behavior_event_types']:
                description['behavior_event_types'][event_type] = 0
            description['behavior_event_types'][event_type] += 1
        
        # Get participant statistics
        participant_event_counts = {}
        all_timestamps = []
        
        # Count events per participant
        for event in self.sensor_events + self.behavior_events:
            relationships = event.get('relationships', [])
            for rel in relationships:
                participant_id = rel.get('objectId')
                if rel.get('qualifier') != 'corr' or not participant_id:
                    continue
                if participant_id not in participant_event_counts:
                    participant_event_counts[participant_id] = {
                        'sensor_events': 0,
                        'behavior_events': 0,
                        'total_events': 0
                    }
                if 'sensorEventType' in event:
                    participant_event_counts[participant_id]['sensor_events'] += 1
                else:
                    participant_event_counts[participant_id]['behavior_events'] += 1
                participant_event_counts[participant_id]['total_events'] += 1
            
            # Collect timestamps
            time_str = event.get('time')
            if time_str:
                try:
                    timestamp = datetime.fromisoformat(time_str.replace('Z', '+00:00'))
                    all_timestamps.append(timestamp)
                except:
                    pass
        
        description['participants'] = participant_event_counts
        
        # Calculate time range
        if all_timestamps:
            earliest = min(all_timestamps)
            latest = max(all_timestamps)
            time_span = (latest - earliest).days
            
            description['time_range'] = {
                'earliest_event': earliest.isoformat(),
                'latest_event': latest.isoformat(),
                'time_span_days': time_span
            }
        
        # Add object type information
        description['object_types'] = [obj_type['name'] for obj_type in self.object_types]
        
        return description
    
    def print_description(self) -> None:
        """
        Print a formatted description of the output data.
        """
        desc = self.describe_output()
        
        print("\n" + "="*70)
        print("OUTPUT DATA DESCRIPTION")
        print("="*70)
        
        print("\n--- SUMMARY ---")
        print(f"Total Events: {desc['summary']['total_events']}")
        print(f"  - Sensor Events: {desc['summary']['total_sensor_events']}")
        print(f"  - Behavior Events: {desc['summary']['total_behavior_events']}")
        print(f"Total Objects: {desc['summary']['total_objects']}")
        print(f"Total Participants: {desc['summary']['total_participants']}")
        
        print("\n--- SENSOR EVENT TYPES ---")
        if desc['sensor_event_types']:
            for event_type, count in sorted(desc['sensor_event_types'].items()):
                print(f"  {event_type}: {count}")
        else:
            print("  No sensor events")
        
        print("\n--- BEHAVIOR EVENT TYPES ---")
        if desc['behavior_event_types']:
            for event_type, count in sorted(desc['behavior_event_types'].items()):
                print(f"  {event_type}: {count}")
        else:
            print("  No behavior events")
        
        print("\n--- TIME RANGE ---")
        if desc['time_range']['earliest_event']:
            print(f"Earliest Event: {desc['time_range']['earliest_event']}")
            print(f"Latest Event: {desc['time_range']['latest_event']}")
            print(f"Time Span: {desc['time_range']['time_span_days']} days")
        else:
            print("  No timestamp data available")
        
        print("\n--- PARTICIPANT STATISTICS ---")
        if desc['participants']:
            for participant_id, stats in sorted(desc['participants'].items()):
                print(f"  Participant {participant_id}:")
                print(f"    - Total Events: {stats['total_events']}")
                print(f"    - Sensor Events: {stats['sensor_events']}")
                print(f"    - Behavior Events: {stats['behavior_events']}")
        else:
            print("  No participant data")
        
        print("\n--- OBJECT TYPES ---")
        for obj_type in desc['object_types']:
            print(f"  {obj_type}")
        
        print("\n" + "="*70 + "\n")
    
    def to_json(self, output_path: Path) -> None:
        """
        Save transformed data to JSON file.
        
        Args:
            output_path: Path to output JSON file
        """
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        ocel = self.transform()
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(ocel, f, indent=2, ensure_ascii=False)
        
        print(f"\nTransformation complete!")
        print(f"Generated:")
        print(f"  - {len(ocel['sensorEvents'])} sensor events")
        print(f"  - {len(ocel['behaviorEvents'])} behavior events")
        print(f"  - {len(ocel['objects'])} objects")
        print(f"\nOutput saved to: {output_path}")


In [8]:
data_dir = Path('../../minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f')
output_path = Path('output/participants_sensor_augmented_ocel-transformed.json')

transformer = DiabetesDatasetTransformer(data_dir)
transformer.to_json(output_path)

# Print detailed description of the output data
transformer.print_description()

Processing demographics...
Processing glucose data...
  -> Loading glucose file: ../../minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2303.csv
  -> Loading glucose file: ../../minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2302.csv
  -> Loading glucose file: ../../minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2405.csv
  -> Loading glucose file: ../../minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2404.csv
  -> Loading glucose file: ../../minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2314.csv
  -> Loading glucose file: ../../minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2308.csv
  -> Loading glucose file: ../../minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2307.csv
  -> Loading glucose file: ../../min